# Agilent BioTek Cytation

The Cytation is an Agilent BioTek multi-mode plate reader with optional microscopy imaging. Depending on the model it supports:

- [Absorbance](../../../capabilities/absorbance)
- [Fluorescence](../../../capabilities/fluorescence)
- [Luminescence](../../../capabilities/luminescence)
- [Microscopy](../../../capabilities/microscopy)
- [Temperature control](../../../capabilities/temperature-control)

| Model | PLR Name | Plate Reading | Microscopy | Temperature |
|---|---|---|---|---|
| Cytation 5 | `Cytation5` | Absorbance, Fluorescence, Luminescence | yes | yes |
| Cytation 1 | `Cytation1` | -- | yes | yes |

Both models use `BioTekBackend` for serial communication and `CytationMicroscopyBackend` for imaging, communicating over FTDI USB. The Cytation 5 adds plate-reading capabilities on top of the shared microscopy and temperature-control features.

## Setup

The examples below use a Cytation 5. For a Cytation 1, replace `Cytation5` with `Cytation1` (the Cytation 1 does not have `.absorbance`, `.fluorescence`, or `.luminescence` attributes).

In [ ]:
from pylabrobot.agilent.biotek import Cytation5

c5 = Cytation5(name="cytation5", device_id="20060813")
await c5.setup()

Open and close the loading tray. Pass `BioTekLoadingTrayBackend.OpenParams(slow=True)` or `CloseParams(slow=True)` for slower motor travel if needed.

In [ ]:
await c5.loading_tray.open()

from pylabrobot.resources import Cor_96_wellplate_360ul_Fb
plate = Cor_96_wellplate_360ul_Fb(name="plate", with_lid=True)
c5.loading_tray.assign_child_resource(plate)

In [ ]:
await c5.loading_tray.close()

## Plate reading (Cytation 5 only)

The Cytation 5 exposes `.absorbance`, `.fluorescence`, and `.luminescence` capability objects. For the full API, see [Absorbance](../../../capabilities/absorbance), [Fluorescence](../../../capabilities/fluorescence), and [Luminescence](../../../capabilities/luminescence).

In [ ]:
# Absorbance
data = await c5.absorbance.read_absorbance(wavelength=450)

In [ ]:
# Fluorescence
data = await c5.fluorescence.read_fluorescence(
    excitation_wavelength=485, emission_wavelength=528, focal_height=7.5
)

In [ ]:
# Luminescence
data = await c5.luminescence.read_luminescence(focal_height=4.5)

## Microscopy

Both the Cytation 5 and Cytation 1 expose a `.microscopy` capability. The Aravis camera is initialized during setup. For the full API, see [Microscopy](../../../capabilities/microscopy).

Use {class}`~pylabrobot.agilent.biotek.plate_readers.cytation.microscopy_backend.CytationMicroscopyBackend.CaptureParams` to control LED intensity and coverage tiling.

In [ ]:
from pylabrobot.agilent.biotek import CytationMicroscopyBackend
from pylabrobot.capabilities.microscopy.standard import ImagingMode, Objective

res = await c5.microscopy.capture(
    well=(1, 2),
    mode=ImagingMode.BRIGHTFIELD,
    objective=Objective.O_4X_PL_FL_Phase,
    focal_height=1.833,
    exposure_time=5,
    gain=8,
    plate=plate,
    backend_params=CytationMicroscopyBackend.CaptureParams(led_intensity=10),
)

In [ ]:
from PIL import Image
Image.fromarray(res.images[0])

Tile multiple fields of view with the `coverage` parameter:

In [ ]:
res = await c5.microscopy.capture(
    well=(1, 2),
    mode=ImagingMode.BRIGHTFIELD,
    objective=Objective.O_4X_PL_FL_Phase,
    focal_height=2,
    exposure_time=5,
    gain=8,
    plate=plate,
    backend_params=CytationMicroscopyBackend.CaptureParams(
        led_intensity=10,
        coverage=(1, 1),
    ),
)
print(f"{len(res.images)} images captured")

from PIL import Image
Image.fromarray(res.images[0])

In [ ]:
# Autofocus
from pylabrobot.capabilities.microscopy import AutoFocus, evaluate_focus_nvmg_sobel

res = await c5.microscopy.capture(
    well=(1, 2),
    mode=ImagingMode.BRIGHTFIELD,
    objective=Objective.O_4X_PL_FL_Phase,
    focal_height=AutoFocus(
        evaluate_focus=evaluate_focus_nvmg_sobel,
        timeout=60,
        low=1,
        high=3,
    ),
    exposure_time=5,
    gain=8,
    plate=plate,
    backend_params=CytationMicroscopyBackend.CaptureParams(led_intensity=10),
)
print(f"best focal height: {res.focal_height:.3f} mm")

from PIL import Image
Image.fromarray(res.images[0])

## Temperature control

Both models expose a `.temperature` controller. For the full API, see [Temperature Control](../../../capabilities/temperature-control).

In [ ]:
await c5.temperature.set_temperature(37.0)

current = await c5.temperature.request_temperature()
print(f"{current:.1f} \u00b0C")

await c5.temperature.deactivate()

## Teardown

In [ ]:
await c5.stop()